# PPE Detector V2 — YOLO26n DetectionHead Study

This notebook studies one focused problem: **helmet and vest boxes can become too small after a construction-site frame is resized to 640 × 640**. It tests whether adding a DetectionHead detection head improves these objects without an unacceptable latency increase.

The four classes remain `helmet`, `no-helmet`, `vest`, and `no-vest`. Dataset, image size, optimizer, augmentation, and seed are kept consistent with the baseline so that the architecture is the main changed variable.

## 1. Why DetectionHead may help

Standard YOLO26n predicts at strides 8, 16, and 32. At a 640-pixel input, its highest-resolution detection map is 80 × 80. YOLO26n DetectionHead adds a stride-4 detection head with a 160 × 160 feature map that retains more spatial detail for distant PPE. The new head layers are initialized through compatible weight transfer from `yolo26n.pt`, then the complete architecture is fine-tuned on Dataset V2.

The hypothesis is accepted only if DetectionHead improves small-object AP while preserving clean validation accuracy and a usable inference speed.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

packages = []
try:
    if Version(version("ultralytics")) < Version("8.4.56"):
        packages.append("ultralytics>=8.4.56")
except PackageNotFoundError:
    packages.append("ultralytics>=8.4.56")
if importlib.util.find_spec("torchmetrics") is None:
    packages.append("torchmetrics")
if importlib.util.find_spec("pycocotools") is None:
    packages.append("pycocotools")
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Environment ready.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from PIL import Image
from IPython.display import display
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO
import ultralytics

DATASET_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
KAGGLE_RESUME_ROOT = Path("/kaggle/input/models/quanghuy225/yolo-p2-resume/pytorch/default/4")

def available_resume_roots():
    if not KAGGLE_RESUME_ROOT.exists():
        return []
    version_dirs = [path for path in KAGGLE_RESUME_ROOT.iterdir() if path.is_dir()]
    version_dirs.sort(key=lambda path: int(path.name) if path.name.isdigit() else -1, reverse=True)
    return version_dirs or [KAGGLE_RESUME_ROOT]

def find_resume_checkpoint(filename):
    for root in available_resume_roots():
        preferred = [
            root / "ppe_detectionhead_study" / "runs" / "E1_detectionhead_seed42_full" / "weights" / filename,
            root / "ppe_detectionhead_study" / "recovery" / filename,
            root / filename,
        ]
        for candidate in preferred:
            if candidate.exists():
                return candidate
        matches = sorted(root.rglob(filename))
        if matches:
            return matches[0]
    return None

_resume_last = find_resume_checkpoint("last.pt")
_resume_best = find_resume_checkpoint("best.pt")
RESUME_WEIGHTS = str(_resume_last) if _resume_last else None
PREVIOUS_BEST_WEIGHTS = str(_resume_best) if _resume_best else None
BASELINE_WEIGHTS = None  # Optional baseline best.pt for direct comparison
RUN_TRAINING = True
EPOCHS = 100
SEED = 42
SAVE_PERIOD = 5  # last.pt still updates every epoch; archive one epoch file every five epochs.
IMAGE_SIZE = 640
PROFILE_IMAGES = 100
CLASS_NAMES = {0: "helmet", 1: "no-helmet", 2: "vest", 3: "no-vest"}
COMPLETE_SOURCES = {"tucje", "gsnvb", "mendeley", "css"}
GPU_COUNT = torch.cuda.device_count()
USE_DUAL_GPU = GPU_COUNT >= 2
TRAIN_DEVICE = [0, 1] if USE_DUAL_GPU else (0 if GPU_COUNT == 1 else "cpu")
EVAL_DEVICE = 0 if GPU_COUNT >= 1 else "cpu"
BATCH = 32 if USE_DUAL_GPU else 16  # Global batch; dual GPU keeps 16 images per T4.
WORKERS = 2 if USE_DUAL_GPU else 4  # Two DDP processes share the Kaggle CPU workers.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Detected GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
print("Training device:", TRAIN_DEVICE, "| Global batch:", BATCH, "| Workers per process:", WORKERS)
print("Resume checkpoint:", RESUME_WEIGHTS or "not found; a fresh DetectionHead run will be used")
print("Previous best checkpoint:", PREVIOUS_BEST_WEIGHTS or "not found")

In [ ]:
def is_dataset_root(path):
    val_name = "valid" if (path / "images" / "valid").exists() else "val"
    return (path / "images" / "train").exists() and (path / "images" / val_name).exists()

def extract_archive(mount):
    archives = sorted(mount.glob("*.rar"))
    if not archives:
        return None
    destination = Path("/kaggle/working/ppe_v2_data")
    destination.mkdir(parents=True, exist_ok=True)
    extractor = shutil.which("7z") or shutil.which("7zz")
    if not extractor:
        raise RuntimeError("Dataset_v2.rar was found but 7-Zip is unavailable. Upload the extracted dataset.")
    subprocess.check_call([extractor, "x", str(archives[0]), f"-o{destination}", "-y"])
    matches = [p for p in destination.rglob("*") if p.is_dir() and is_dataset_root(p)]
    return matches[0].resolve() if matches else None

def find_dataset_root():
    if DATASET_ROOT_OVERRIDE:
        root = Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
        if is_dataset_root(root):
            return root
    mounts = [
        Path("/kaggle/input/datasets/linhlaz225/helmet-nohelmet-vest-novest"),
        Path("/kaggle/input/helmet-nohelmet-vest-novest"),
    ]
    local_candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        local_candidates.extend([parent / "Dataset_v2", parent / "Dataset" / "Dataset_v2"])
    for candidate in [*(m / "Dataset_v2" for m in mounts), *mounts, *local_candidates]:
        if is_dataset_root(candidate):
            return candidate.resolve()
    for mount in mounts:
        if mount.exists():
            nested = [p for p in mount.rglob("*") if p.is_dir() and is_dataset_root(p)]
            if nested:
                return nested[0].resolve()
            extracted = extract_archive(mount)
            if extracted:
                return extracted
    raise FileNotFoundError("Dataset V2 was not found. Set DATASET_ROOT_OVERRIDE if necessary.")

DATASET_ROOT = find_dataset_root()
VAL_FOLDER = "valid" if (DATASET_ROOT / "images" / "valid").exists() else "val"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).resolve() if OUTPUT_ROOT_OVERRIDE else (
    Path("/kaggle/working/ppe_detectionhead_study") if Path("/kaggle/working").exists()
    else Path.cwd() / "ai-model" / "outputs" / "ppe_detectionhead_study"
)
RUNS_DIR = OUTPUT_ROOT / "runs"
RESULTS_DIR = OUTPUT_ROOT / "results"
CONFIG_DIR = OUTPUT_ROOT / "config"
RECOVERY_DIR = OUTPUT_ROOT / "recovery"
EXPORT_DIR = OUTPUT_ROOT / "export"
for folder in [RUNS_DIR, RESULTS_DIR, CONFIG_DIR, RECOVERY_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

data_config = {
    "path": str(DATASET_ROOT), "train": "images/train", "val": f"images/{VAL_FOLDER}",
    "test": "images/test", "nc": 4, "names": CLASS_NAMES,
}
DATA_YAML = CONFIG_DIR / "dataset_v2.yaml"
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print("Dataset:", DATASET_ROOT)
print("Outputs:", OUTPUT_ROOT)

## 2. Verify the architectural change

This cell confirms that DetectionHead adds stride 4 while the baseline begins at stride 8. Parameter count is reported because additional high-resolution computation affects edge deployment.

In [ ]:
def architecture_record(config):
    model = YOLO(config)
    strides = [int(value) for value in model.model.stride.tolist()]
    return {
        "architecture": config,
        "strides": strides,
        "feature_maps_at_640": [IMAGE_SIZE // stride for stride in strides],
        "parameters": sum(parameter.numel() for parameter in model.model.parameters()),
    }

architecture_table = pd.DataFrame([architecture_record("yolo26n.yaml"), architecture_record("yolo26n-p2.yaml")])
architecture_table["parameter_increase_percent"] = (architecture_table["parameters"] / architecture_table.loc[0, "parameters"] - 1) * 100
display(architecture_table)
assert architecture_table.loc[0, "strides"] == [8, 16, 32]
assert architecture_table.loc[1, "strides"] == [4, 8, 16, 32]

## 3. Measure the small-object problem

Bounding boxes are projected to a 640 × 640 input. Boxes below 32² pixels follow the common small-object threshold; the additional `tiny` group below 16² pixels is used only to describe the dataset.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_rows, box_rows = [], []
for split, folder in {"train": "train", "val": VAL_FOLDER}.items():
    image_dir = DATASET_ROOT / "images" / folder
    label_dir = DATASET_ROOT / "labels" / folder
    for image_path in sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS):
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            raise FileNotFoundError(f"Missing label: {label_path}")
        source = image_path.stem.split("__", 1)[0].lower()
        image_rows.append({"split": split, "source": source, "image_path": str(image_path.resolve()), "label_path": str(label_path.resolve())})
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f"Invalid YOLO label at {label_path}:{line_number}")
            class_id = int(float(values[0]))
            x, y, width, height = map(float, values[1:])
            area = width * height * IMAGE_SIZE * IMAGE_SIZE
            size_group = "tiny" if area < 16**2 else "small" if area < 32**2 else "medium_or_large"
            box_rows.append({
                "split": split, "source": source, "image_path": str(image_path.resolve()),
                "label_path": str(label_path.resolve()), "class_id": class_id,
                "class_name": CLASS_NAMES[class_id], "x": x, "y": y, "w": width, "h": height,
                "area_at_640": area, "size_group": size_group,
            })

images_df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)
size_table = boxes_df.groupby(["class_name", "size_group"]).size().unstack(fill_value=0).reindex(CLASS_NAMES.values())
size_table["tiny_or_small_ratio"] = (size_table.get("tiny", 0) + size_table.get("small", 0)) / size_table.sum(axis=1).clip(lower=1)
display(size_table)
size_table.to_csv(RESULTS_DIR / "object_size_distribution.csv")
plt.figure(figsize=(9, 4))
sns.ecdfplot(data=boxes_df[boxes_df["split"] == "train"], x="area_at_640", hue="class_name", log_scale=True)
plt.axvline(16**2, color="black", linestyle=":", linewidth=1)
plt.axvline(32**2, color="black", linestyle="--", linewidth=1)
plt.title("Object area after resize to 640 × 640")
plt.xlabel("Bounding-box area in pixels")
plt.tight_layout()
plt.show()

## 4. Train YOLO26n DetectionHead

The recipe matches the baseline. Only the architecture changes from `yolo26n.pt` to `yolo26n-p2.yaml`; compatible baseline pretrained weights are transferred before fine-tuning. Training resumes from `last.pt` when available. On Kaggle T4 ×2, the notebook uses Distributed Data Parallel with a global batch of 32. Each GPU processes 16 images, while `nbs=64` preserves the same nominal effective-batch target used by Ultralytics. `last.pt` is used only for continuation; `best.pt` is the strongest validation checkpoint and is carried across Kaggle Versions before resume.

In [ ]:
RUN_NAME = "E1_detectionhead_seed42_full"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

def completed_epochs():
    return len(pd.read_csv(RESULTS_CSV)) if RESULTS_CSV.exists() else 0

def stage_resume_files(source_last):
    source_last = Path(source_last).expanduser().resolve()
    if not source_last.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {source_last}")
    LAST_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    if source_last != LAST_WEIGHTS.resolve():
        shutil.copy2(source_last, LAST_WEIGHTS)

    best_candidates = []
    if PREVIOUS_BEST_WEIGHTS:
        best_candidates.append(Path(PREVIOUS_BEST_WEIGHTS).expanduser())
    best_candidates.append(source_last.with_name("best.pt"))
    previous_best = next((path for path in best_candidates if path.exists()), None)
    if previous_best:
        if previous_best.resolve() != BEST_WEIGHTS.resolve():
            shutil.copy2(previous_best, BEST_WEIGHTS)
        print("Preserved previous strongest checkpoint:", previous_best)
    else:
        print("Warning: previous best.pt was not found. Add the full weights folder or set PREVIOUS_BEST_WEIGHTS.")

    history_candidates = [source_last.parent / "results.csv", source_last.parent.parent / "results.csv"]
    previous_history = next((path for path in history_candidates if path.exists()), None)
    if previous_history and previous_history.resolve() != RESULTS_CSV.resolve():
        shutil.copy2(previous_history, RESULTS_CSV)
    return LAST_WEIGHTS

def save_recovery(trainer):
    RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
    for source in [Path(trainer.last), Path(trainer.best), Path(trainer.csv), Path(trainer.save_dir) / "args.yaml"]:
        if source.exists():
            shutil.copy2(source, RECOVERY_DIR / source.name)
    state = {"experiment": "E1_detectionhead", "completed_epochs": int(trainer.epoch) + 1, "updated_utc": datetime.now(timezone.utc).isoformat()}
    (RECOVERY_DIR / "state.json").write_text(json.dumps(state, indent=2), encoding="utf-8")

train_args = {
    "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH,
    "workers": WORKERS, "device": TRAIN_DEVICE, "seed": SEED, "deterministic": True,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.01, "cos_lr": True,
    "warmup_epochs": 3, "weight_decay": 0.0005, "hsv_h": 0.015,
    "hsv_s": 0.55, "hsv_v": 0.35, "degrees": 5, "translate": 0.10,
    "scale": 0.45, "fliplr": 0.5, "mosaic": 0.5, "mixup": 0.05,
    "close_mosaic": 10, "amp": True, "nbs": 64, "project": str(RUNS_DIR), "name": RUN_NAME,
    "exist_ok": True, "save": True, "save_period": SAVE_PERIOD, "val": True,
    "patience": 100, "plots": True,
}
manifest = {
    "experiment": "E1_detectionhead", "architecture": "yolo26n-p2.yaml",
    "pretrained_transfer": "yolo26n.pt", "seed": SEED, "classes": CLASS_NAMES,
    "dataset": str(DATASET_ROOT), "training": train_args,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__},
}
(RESULTS_DIR / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

if RUN_TRAINING:
    resume_path = Path(RESUME_WEIGHTS).expanduser() if RESUME_WEIGHTS else LAST_WEIGHTS
    if completed_epochs() >= EPOCHS and BEST_WEIGHTS.exists():
        print(f"DetectionHead training already complete: {completed_epochs()}/{EPOCHS} epochs")
    elif resume_path.exists():
        resume_path = stage_resume_files(resume_path)
        print("Resuming DetectionHead from staged checkpoint:", resume_path)
        model = YOLO(str(resume_path))
        model.add_callback("on_model_save", save_recovery)
        model.train(resume=True, data=str(DATA_YAML), device=TRAIN_DEVICE, workers=WORKERS, batch=BATCH, save_period=SAVE_PERIOD, plots=True)
    else:
        model = YOLO("yolo26n-p2.yaml")
        model.load("yolo26n.pt")
        model.add_callback("on_model_save", save_recovery)
        model.train(**train_args)
else:
    print("Training disabled.")

## 5. Clean and small-object evaluation

Clean validation measures overall detection quality. Small-object AP is computed separately because it is the direct reason for adding DetectionHead. For this measurement, validation images from complete four-label sources are preferred to avoid treating unannotated PPE as false positives.

In [ ]:
ACTIVE_BEST_WEIGHTS = BEST_WEIGHTS if BEST_WEIGHTS.exists() else RECOVERY_DIR / "best.pt"
ACTIVE_RESULTS_CSV = RESULTS_CSV if RESULTS_CSV.exists() else RECOVERY_DIR / "results.csv"
if not ACTIVE_BEST_WEIGHTS.exists():
    raise FileNotFoundError("DetectionHead best.pt was not found in the run or recovery folder.")

def clean_metrics(weights, name):
    metrics = YOLO(str(weights)).val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=1, device=EVAL_DEVICE, plots=name == "E1_detectionhead")
    box = metrics.box
    record = {"experiment": name, "map50_95": float(box.map), "map50": float(box.map50), "precision_macro": float(box.mp), "recall_macro": float(box.mr)}
    class_ids = [int(value) for value in np.asarray(box.ap_class_index).tolist()]
    for index, class_id in enumerate(class_ids):
        record[f"{CLASS_NAMES[class_id]}_ap50_95"] = float(np.asarray(box.ap)[index])
        record[f"{CLASS_NAMES[class_id]}_recall"] = float(np.asarray(box.r)[index])
    return record

IMAGE_TO_LABEL = dict(zip(images_df["image_path"], images_df["label_path"]))
def target_for_image(image_path):
    image_path = str(Path(image_path).resolve())
    with Image.open(image_path) as image:
        image_width, image_height = image.size
    boxes, labels = [], []
    for line in Path(IMAGE_TO_LABEL[image_path]).read_text(encoding="utf-8", errors="ignore").splitlines():
        class_id, x, y, width, height = map(float, line.split())
        boxes.append([(x - width / 2) * image_width, (y - height / 2) * image_height, (x + width / 2) * image_width, (y + height / 2) * image_height])
        labels.append(int(class_id))
    return {"boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), "labels": torch.tensor(labels, dtype=torch.int64)}

def small_object_metrics(weights, name):
    validation = images_df[(images_df["split"] == "val") & images_df["source"].isin(COMPLETE_SOURCES)]
    if validation.empty:
        print("Complete-source prefixes were not found; using the full validation split.")
        validation = images_df[images_df["split"] == "val"]
    model = YOLO(str(weights))
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    paths = validation["image_path"].tolist()
    for start in range(0, len(paths), 16):
        batch = paths[start:start + 16]
        predictions = model.predict(batch, imgsz=IMAGE_SIZE, conf=0.001, iou=0.7, device=EVAL_DEVICE, verbose=False)
        predicted = [{
            "boxes": result.boxes.xyxy.detach().cpu().float(),
            "scores": result.boxes.conf.detach().cpu().float(),
            "labels": result.boxes.cls.detach().cpu().long(),
        } for result in predictions]
        metric.update(predicted, [target_for_image(path) for path in batch])
    values = metric.compute()
    return {"experiment": name, "ap_small": float(values["map_small"]), "ap_medium": float(values["map_medium"]), "ap_large": float(values["map_large"])}

models = [("E1_detectionhead", ACTIVE_BEST_WEIGHTS)]
if BASELINE_WEIGHTS and Path(BASELINE_WEIGHTS).expanduser().exists():
    models.insert(0, ("E0_baseline", Path(BASELINE_WEIGHTS).expanduser()))
else:
    print("Set BASELINE_WEIGHTS to compare E0 and E1 in the same evaluation.")
clean_df = pd.DataFrame([clean_metrics(weights, name) for name, weights in models])
small_df = pd.DataFrame([small_object_metrics(weights, name) for name, weights in models])
evaluation_df = clean_df.merge(small_df, on="experiment")
evaluation_df.to_csv(RESULTS_DIR / "detectionhead_validation.csv", index=False)
display(evaluation_df)

## 6. Accuracy–speed decision

DetectionHead is useful only when its small-object gain justifies its additional computation. The following profile measures both models on the same device and image sample when baseline weights are supplied. Kaggle timing is suitable for comparison, but the final deployment speed must still be measured on the target Jetson.

In [ ]:
def profile_model(weights, name):
    validation = images_df[images_df["split"] == "val"].sample(min(PROFILE_IMAGES, sum(images_df["split"] == "val")), random_state=SEED)
    paths = validation["image_path"].tolist()
    model = YOLO(str(weights))
    for path in paths[:10]:
        model.predict(path, imgsz=IMAGE_SIZE, device=EVAL_DEVICE, verbose=False)
    elapsed = []
    for path in paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        model.predict(path, imgsz=IMAGE_SIZE, device=EVAL_DEVICE, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append((time.perf_counter() - start) * 1000)
    p50 = float(np.percentile(elapsed, 50))
    return {
        "experiment": name, "parameters": sum(p.numel() for p in model.model.parameters()),
        "weight_size_mb": Path(weights).stat().st_size / 1024**2, "latency_p50_ms": p50,
        "latency_p95_ms": float(np.percentile(elapsed, 95)), "fps_from_p50": 1000 / max(p50, 1e-9),
    }

profile_df = pd.DataFrame([profile_model(weights, name) for name, weights in models])
comparison = evaluation_df.merge(profile_df, on="experiment")
if "E0_baseline" in set(comparison["experiment"]):
    baseline = comparison[comparison["experiment"] == "E0_baseline"].iloc[0]
    comparison["small_ap_gain_vs_e0"] = comparison["ap_small"] - baseline["ap_small"]
    comparison["map50_95_change_vs_e0"] = comparison["map50_95"] - baseline["map50_95"]
    comparison["latency_ratio_vs_e0"] = comparison["latency_p50_ms"] / baseline["latency_p50_ms"]
comparison.to_csv(RESULTS_DIR / "detectionhead_accuracy_speed_comparison.csv", index=False)
display(comparison)

history = pd.read_csv(ACTIVE_RESULTS_CSV)
history.columns = history.columns.str.strip()
metric_column = "metrics/mAP50-95(B)"
best_row = history.loc[history[metric_column].idxmax()]
print("Best DetectionHead epoch:", int(best_row["epoch"]) + 1)
print("Highest recorded validation mAP50-95:", round(float(best_row[metric_column]), 4))

## 7. Output

The study produces DetectionHead `best.pt`, clean metrics, small-object AP, latency, and the architecture audit. Select DetectionHead only after comparing these values with the separately trained baseline.

In [ ]:
exported = EXPORT_DIR / "ppe_v2_yolo26n_detectionhead_best.pt"
shutil.copy2(ACTIVE_BEST_WEIGHTS, exported)
print("DetectionHead weights:", exported)
print("Evaluation:", RESULTS_DIR / "detectionhead_validation.csv")
print("Accuracy-speed comparison:", RESULTS_DIR / "detectionhead_accuracy_speed_comparison.csv")